In [0]:
from pyspark.sql import functions as f
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from dbx_utils.feature_flag import feature_flag

FEDERAL_BRACKETS_2026 = [(58_523, "14%"), (117_045, "20.5%"), (181_440, "26%"), (258_482, "29%")]
TOP_RATE = "33%"

df = spark.createDataFrame(
    [
        (1, "Alice", "Toronto", 34, 85000.0, "2024-01-15"),
        (2, "Bob", "Montreal", 28, 62000.0, "2024-02-03"),
        (3, "Carol", "Vancouver", 45, 120000.0, "2023-11-20"),
        (3, "Carol", "Vancouver", 45, 120000.0, "2023-11-20"),
        (4, "Dave", "Calgary", 39, 74500.0, "2024-03-09"),
        (5, "Eve", "Ottawa", 31, 91000.0, "2024-04-22")],
    schema="id INT, name STRING, city STRING, age INT, salary DOUBLE, hire_date STRING")

In [0]:
# Toggle these flags to see how the transformations are applied.
# Ideally these are job or task parameters which depend on the
# environment the job is running in.
NEW_CASTING = True
ADDITIONAL_GOVERANCE = True

def old_casting_transformation(df):
    return (df
        .withColumn("hire_date", f.col("hire_date").cast("timestamp"))
        .select("*"))

@feature_flag(NEW_CASTING, fallback=old_casting_transformation)
def new_casting_transformation(df):
    return (df
        .withColumn("hire_date", f.col("hire_date").cast("date"))
        .select("*"))

def cur_transformation(df):
    col = f.col("salary")
    expr = f.when(col.isNull(), f.lit(None))
    for upper, rate in FEDERAL_BRACKETS_2026:
        expr = expr.when(col <= upper, f.lit(rate))
    return (df
        .withColumn("tax_bracket", expr.otherwise(f.lit(TOP_RATE)))
        .drop("salary"))

@feature_flag(ADDITIONAL_GOVERANCE)
def new_transformation(df):
    lo = f.floor(f.col("age") / 10) * 10
    return (df
        .withColumn("age_bracket", f.format_string("[%d-%d]", lo, lo + 10))
        .drop("age"))

final_df = (df
    .transform(cur_transformation)
    .transform(new_transformation)
    .transform(new_casting_transformation)
)

final_df.display()